In [1]:
%pip install -q langchain langchain-core langchain-community langchain-groq python-dotenv requests

Note: you may need to restart the kernel to use updated packages.


In [4]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv
import os
import requests

In [5]:
load_dotenv()

True

In [7]:
# tool create
@tool
def multiply(a: int, b: int) -> int:
  """Given 2 numbers a and b this tool returns their product"""
  return a * b

In [8]:
print(multiply.invoke({'a':3, 'b':4}))

12


In [9]:
multiply.name

'multiply'

In [10]:
multiply.description

'Given 2 numbers a and b this tool returns their product'

In [11]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

In [ ]:
llm = ChatGroq(
    model="llama-3.1-8b-instant"
)

In [17]:
llm.invoke('hi')

AIMessage(content='Hello! Is there something I can help you with or would you like to chat?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 36, 'total_tokens': 54, 'completion_time': 0.035193573, 'completion_tokens_details': None, 'prompt_time': 0.019197358, 'prompt_tokens_details': None, 'queue_time': 0.012907989, 'total_time': 0.054390931}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f92c2-e28e-7e20-a3d3-53aad5e200bc-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 36, 'output_tokens': 18, 'total_tokens': 54})

In [ ]:
# tool binding
llm_with_tools = llm.bind_tools([multiply])

In [19]:
llm_with_tools.invoke('Hi how are you')

AIMessage(content="I'm functioning properly, thank you for asking. What would you like to talk about or ask assistance with?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 234, 'total_tokens': 257, 'completion_time': 0.044839396, 'completion_tokens_details': None, 'prompt_time': 0.015141896, 'prompt_tokens_details': None, 'queue_time': 0.005890993, 'total_time': 0.059981292}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f92c2-f646-7cc0-9627-143a99d27384-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 234, 'output_tokens': 23, 'total_tokens': 257})

In [20]:
query = HumanMessage('can you multiply 3 with 1000')

In [21]:
messages = [query]

In [22]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={})]

In [23]:
result = llm_with_tools.invoke(messages)

In [24]:
messages.append(result)

In [25]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'ntdm2w1gx', 'function': {'arguments': '{"a":3,"b":1000}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 239, 'total_tokens': 259, 'completion_time': 0.031231721, 'completion_tokens_details': None, 'prompt_time': 0.022519447, 'prompt_tokens_details': None, 'queue_time': 0.006520698, 'total_time': 0.053751168}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f92c3-3654-77a0-b13e-fcf1a76e7cde-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'ntdm2w1gx', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 239, 'output_tokens': 20, 'total_tokens': 259})]

In [26]:
tool_result = multiply.invoke(result.tool_calls[0])

In [27]:
tool_result

ToolMessage(content='3000', name='multiply', tool_call_id='ntdm2w1gx')

In [28]:
messages.append(tool_result)

In [29]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'ntdm2w1gx', 'function': {'arguments': '{"a":3,"b":1000}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 239, 'total_tokens': 259, 'completion_time': 0.031231721, 'completion_tokens_details': None, 'prompt_time': 0.022519447, 'prompt_tokens_details': None, 'queue_time': 0.006520698, 'total_time': 0.053751168}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f92c3-3654-77a0-b13e-fcf1a76e7cde-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'ntdm2w1gx', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 239, 'output_tokens': 20, 'total_tokens': 259}),
 ToolMe

In [31]:
llm_with_tools.invoke(messages).content

'Alternatively, I can provide the result directly: 3000'

In [48]:
# tool create
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate


In [33]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'}}

In [34]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1784851201,
 'time_last_update_utc': 'Fri, 24 Jul 2026 00:00:01 +0000',
 'time_next_update_unix': 1784937601,
 'time_next_update_utc': 'Sat, 25 Jul 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 96.7306}

In [35]:
convert.invoke({'base_currency_value':10, 'conversion_rate':85.16})

851.5999999999999

In [ ]:
llm = ChatGroq(
    model="llama-3.1-8b-instant"
)

In [ ]:
# tool binding
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [38]:
messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd')]

In [49]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '39xjccm8d', 'function': {'arguments': '{"base_currency":"INR","target_currency":"USD"}', 'name': 'get_conversion_factor'}, 'type': 'function'}, {'id': 'xdjnr2wv8', 'function': {'arguments': '{"base_currency_value":10,"conversion_factor":0.0137}', 'name': 'convert'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 350, 'total_tokens': 398, 'completion_time': 0.07583361, 'completion_tokens_details': None, 'prompt_time': 0.02209843, 'prompt_tokens_details': None, 'queue_time': 0.005958368, 'total_time': 0.09793204}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--0

In [50]:
ai_message = llm_with_tools.invoke(messages)

In [51]:
messages.append(ai_message)

In [42]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'INR', 'target_currency': 'USD'},
  'id': '39xjccm8d',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10, 'conversion_factor': 0.0137},
  'id': 'xdjnr2wv8',
  'type': 'tool_call'}]

In [43]:
import json

for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)



In [44]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '39xjccm8d', 'function': {'arguments': '{"base_currency":"INR","target_currency":"USD"}', 'name': 'get_conversion_factor'}, 'type': 'function'}, {'id': 'xdjnr2wv8', 'function': {'arguments': '{"base_currency_value":10,"conversion_factor":0.0137}', 'name': 'convert'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 350, 'total_tokens': 398, 'completion_time': 0.07583361, 'completion_tokens_details': None, 'prompt_time': 0.02209843, 'prompt_tokens_details': None, 'queue_time': 0.005958368, 'total_time': 0.09793204}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--0

In [45]:
llm_with_tools.invoke(messages).content

'The conversion factor between INR and USD is 1 USD = 96.25 INR.\n10 INR is equal to 0.1034 USD.'

In [53]:
# from langchain.agents import initialize_agent, AgentType

# # Step 5: Initialize the Agent ---
# agent_executor = initialize_agent(
#     tools=[get_conversion_factor, convert],
#     llm=llm,
#     agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,  # using ReAct pattern
#     verbose=True  # shows internal thinking
# )
# # --- Step 6: Run the Agent ---
# user_query = "Hi how are you?"

# response = agent_executor.invoke({"input": user_query})